#KHỞI TẠO MÔI TRƯỜNG VÀ DỮ LIỆU

##Tạo Database và dữ liệu mẫu

In [1]:
import sqlite3
import pandas as pd

# 1. Tạo kết nối đến database trong bộ nhớ (RAM)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Tạo bảng và chèn dữ liệu
cursor.executescript("""
    CREATE TABLE course (
        id INTEGER PRIMARY KEY,
        course_name TEXT
    );

    CREATE TABLE student (
        student_id INTEGER PRIMARY KEY,
        name TEXT,
        class TEXT,
        course_id INTEGER,
        score REAL
    );

    INSERT INTO course (id, course_name) VALUES
    (12, 'Giai tich'), (34, 'Thong ke'), (26, 'Tin hoc');

    INSERT INTO student (student_id, name, class, course_id, score) VALUES
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', NULL, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', NULL, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', NULL, 7.0);
""")
conn.commit()

# 3. Hàm hỗ trợ in kết quả SQL ra bảng Pandas Dataframe cho đẹp mắt
def run_query(query, title=""):
    if title:
        print(f"\n{title}")
    display(pd.read_sql_query(query, conn))

print("Đã khởi tạo database và dữ liệu thành công!")

Đã khởi tạo database và dữ liệu thành công!


Kết luận: Môi trường đã sẵn sàng. Dữ liệu gốc gồm 10 sinh viên (trong đó có sinh viên thiếu course_id hoặc mang course_id ảo như 20, 24) và 3 môn học.

#KẾT NỐI HAI BẢNG

##1. Tích Descartes (Cross Join)

In [2]:
run_query("SELECT * FROM student CROSS JOIN course LIMIT 5;", "1.1. Tích Descartes (CROSS JOIN):")


1.1. Tích Descartes (CROSS JOIN):


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34,9.2,26,Tin hoc


Kết luận: Kết quả hiển thị sự kết hợp của mỗi sinh viên với TẤT CẢ các môn học có trong bảng course. Nếu in hết, bảng sẽ có 30 dòng (10 sinh viên × 3 môn học). Phép toán này ít dùng trong thực tế trừ khi muốn tạo tổ hợp.

##2. Inner Join

In [3]:
run_query("SELECT s.*, c.course_name FROM student s INNER JOIN course c ON s.course_id = c.id;", "1.2. INNER JOIN:")


1.2. INNER JOIN:


,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke


Kết luận: Chỉ hiển thị những sinh viên có course_id khớp chính xác với id trong bảng course (chỉ có sinh viên học môn 12 và 34). Những sinh viên có course_id là NULL, 20, hoặc 24 đều bị loại bỏ khỏi bảng kết quả.

##3. Left Join

In [4]:
run_query("SELECT s.*, c.course_name FROM student s LEFT JOIN course c ON s.course_id = c.id;", "1.3. LEFT JOIN:")


1.3. LEFT JOIN:


,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


Kết luận: Giữ lại TOÀN BỘ 10 sinh viên. Những sinh viên không có môn học hợp lệ (NULL, 20, 24) vẫn xuất hiện, nhưng cột course_name của họ sẽ bị bỏ trống (hiển thị là None hoặc NaN trong Pandas).

##4. Full Outer Join

In [5]:
run_query("""
    SELECT s.student_id, s.name, s.course_id, c.id, c.course_name
    FROM student s LEFT JOIN course c ON s.course_id = c.id
    UNION
    SELECT s.student_id, s.name, s.course_id, c.id, c.course_name
    FROM course c LEFT JOIN student s ON c.id = s.course_id;
""", "1.4. FULL OUTER JOIN:")


1.4. FULL OUTER JOIN:


,student_id,name,course_id,id,course_name
0,NaN,None,NaN,26.0,Tin hoc
1,1.0,Nguyen Minh Hoang,12.0,12.0,Giai tich
2,2.0,Tran Thi Lan,34.0,34.0,Thong ke
3,3.0,Pham Van Nam,NaN,NaN,None
4,4.0,Le Thanh Huyen,20.0,NaN,None
5,5.0,Vu Quoc Anh,24.0,NaN,None
6,6.0,Dang Thuy Linh,24.0,NaN,None
7,7.0,Bui Tien Dung,34.0,34.0,Thong ke
8,8.0,Ho Ngoc Mai,20.0,NaN,None
9,9.0,Duong Huu Phuc,NaN,NaN,None


Kết luận: Kết hợp cả Left Join và Right Join. Kết quả hiển thị toàn bộ sinh viên (kể cả không có môn học) và toàn bộ môn học (kể cả môn không có sinh viên nào đăng ký - môn Tin học mã 26).

#CẬP NHẬT & THỐNG KÊ

##1. Cập nhật (UPDATE) và Xóa (DELETE)

In [6]:
cursor.executescript("""
    UPDATE student SET course_id = 26 WHERE course_id IS NULL;
    DELETE FROM student WHERE course_id NOT IN (SELECT id FROM course);
""")
conn.commit()
run_query("SELECT * FROM student;", "2.1. Dữ liệu sau khi Cập nhật và Xóa:")


2.1. Dữ liệu sau khi Cập nhật và Xóa:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,26,7.9
3,7,Bui Tien Dung,Kinh Te,34,9.2
4,9,Duong Huu Phuc,Kinh Te,26,7.2
5,10,Cao Thi Hanh,May Tinh,26,7.0


Kết luận: Các sinh viên bị thiếu môn (NULL) đã được gắn mã 26 (Tin học). Các sinh viên mang mã ảo 20 và 24 đã bị xóa hoàn toàn khỏi cơ sở dữ liệu. Tổng số lượng sinh viên hiện tại đã giảm đi.

##2. Thống kê theo Lớp

In [7]:
run_query("""
    SELECT class, COUNT(student_id) AS Tong_SV, ROUND(AVG(score), 2) AS Diem_TB
    FROM student GROUP BY class;
""", "2.2. Tổng số SV và Điểm TB từng Lớp:")


2.2. Tổng số SV và Điểm TB từng Lớp:


,class,Tong_SV,Diem_TB
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


Kết luận: Nhóm các sinh viên theo lớp (Kinh Te, May Tinh, Toan Tin). Ta thấy rõ lớp nào đông nhất và lớp nào có chất lượng học tập (Điểm TB) cao nhất dựa trên dữ liệu đã làm sạch.

##3. Thống kê theo Môn học

In [8]:
run_query("""
    SELECT c.course_name, COUNT(s.student_id) AS Tong_SV, ROUND(AVG(s.score), 2) AS Diem_TB
    FROM student s INNER JOIN course c ON s.course_id = c.id GROUP BY c.course_name;
""", "2.3. Tổng số SV và Điểm TB từng Môn học:")


2.3. Tổng số SV và Điểm TB từng Môn học:


,course_name,Tong_SV,Diem_TB
0,Giai tich,1,6.70
1,Thong ke,2,9.20
2,Tin hoc,3,7.37


Kết luận: Phép kết nối (INNER JOIN) kết hợp với GROUP BY giúp ta biết được mỗi môn học có bao nhiêu sinh viên đang theo học và điểm trung bình của môn đó là bao nhiêu. Môn Tin học (do ta vừa gán vào) giờ đã có dữ liệu.

##4. Phân loại thi đua

In [9]:
run_query("""
    SELECT c.course_name, ROUND(AVG(s.score), 2) AS Diem_TB,
        CASE
            WHEN AVG(s.score) >= 9.0 THEN 'Xuất sắc'
            WHEN AVG(s.score) >= 6.0 THEN 'Tốt'
            ELSE 'Kém'
        END AS Phan_Loai
    FROM student s INNER JOIN course c ON s.course_id = c.id GROUP BY c.course_name;
""", "2.4. Phân loại thi đua theo Môn học:")


2.4. Phân loại thi đua theo Môn học:


,course_name,Diem_TB,Phan_Loai
0,Giai tich,6.70,Tốt
1,Thong ke,9.20,Xuất sắc
2,Tin hoc,7.37,Tốt


Kết luận: Bảng sử dụng lệnh CASE WHEN để tự động dán nhãn chất lượng cho từng môn học dựa trên quy tắc điểm số. Ta thấy trực quan môn nào đạt danh hiệu Xuất sắc, Tốt hay Kém.

#XẾP HẠNG SINH VIÊN

##1.Xếp hạng tổng thể (Tạo View)

In [10]:
cursor.execute("""
    CREATE VIEW RankedStudents AS
    SELECT student_id, name, class, course_id, score,
        DENSE_RANK() OVER (ORDER BY score DESC) AS rank_overall
    FROM student;
""")
conn.commit()
run_query("SELECT * FROM RankedStudents;", "3.1. Bảng Xếp hạng Toàn trường:")


3.1. Bảng Xếp hạng Toàn trường:


,student_id,name,class,course_id,score,rank_overall
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,2
3,9,Duong Huu Phuc,Kinh Te,26,7.2,3
4,10,Cao Thi Hanh,May Tinh,26,7.0,4
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,5


Kết luận: Lệnh DENSE_RANK() đã xếp hạng sinh viên từ điểm cao nhất xuống thấp nhất. Những sinh viên có điểm bằng nhau (ví dụ cùng 9.2) sẽ đồng hạng 1, và người tiếp theo sẽ là hạng 2 (không bị bỏ cóc thứ hạng).

##2. Top 3 và Bottom 3

In [11]:
run_query("SELECT name, score, rank_overall FROM RankedStudents WHERE rank_overall <= 3;", "TOP 3 TOÀN TRƯỜNG:")

run_query("""
    SELECT name, score, rank_overall FROM RankedStudents
    WHERE rank_overall IN (SELECT DISTINCT rank_overall FROM RankedStudents ORDER BY rank_overall DESC LIMIT 3)
    ORDER BY rank_overall DESC;
""", "BOTTOM 3 TOÀN TRƯỜNG:")


TOP 3 TOÀN TRƯỜNG:


,name,score,rank_overall
0,Tran Thi Lan,9.2,1
1,Bui Tien Dung,9.2,1
2,Pham Van Nam,7.9,2
3,Duong Huu Phuc,7.2,3



BOTTOM 3 TOÀN TRƯỜNG:


,name,score,rank_overall
0,Nguyen Minh Hoang,6.7,5
1,Cao Thi Hanh,7.0,4
2,Duong Huu Phuc,7.2,3


Kết luận: Dựa vào cột rank_overall đã tính toán ở trên, ta lọc ra được chính xác nhóm sinh viên xuất sắc nhất và nhóm sinh viên cần cố gắng nhất trong toàn trường. Việc dùng View giúp truy vấn ngắn gọn hơn.

#NGÀY TỐT NGHIỆP VÀ ĐÓNG KẾT NỐI

##1. Thêm ngày tốt nghiệp

In [12]:
cursor.executescript("""
    ALTER TABLE student ADD COLUMN graduation_date DATETIME;
    UPDATE student
    SET graduation_date = datetime('now', '+' || (
        SELECT DENSE_RANK() OVER (ORDER BY score DESC)
        FROM student AS s2 WHERE s2.student_id = student.student_id
    ) || ' days');
""")
conn.commit()

run_query("SELECT student_id, name, score, graduation_date FROM student ORDER BY score DESC;", "4. Ngày tốt nghiệp dự kiến:")

# Đóng kết nối DB
conn.close()


4. Ngày tốt nghiệp dự kiến:


,student_id,name,score,graduation_date
0,2,Tran Thi Lan,9.2,2026-04-04 03:46:47
1,7,Bui Tien Dung,9.2,2026-04-04 03:46:47
2,3,Pham Van Nam,7.9,2026-04-04 03:46:47
3,9,Duong Huu Phuc,7.2,2026-04-04 03:46:47
4,10,Cao Thi Hanh,7.0,2026-04-04 03:46:47
5,1,Nguyen Minh Hoang,6.7,2026-04-04 03:46:47


Kết luận: Bảng kết quả hiển thị thêm một cột thời gian thực (Datetime). Bằng cách kết hợp hàm datetime('now') của SQLite và thứ hạng (rank) quy đổi thành số ngày (days), những sinh viên hạng cao (hạng 1) sẽ có ngày tốt nghiệp sớm nhất (ví dụ: ngày mai), và lùi dần đối với các sinh viên xếp hạng thấp hơn. Cơ sở dữ liệu sau đó được đóng lại an toàn.